# CineMovie — Recommendation Model

## 1. TF-IDF Vectorization

In [1]:
# Import Pandas for working with our movie dataframe.
import pandas as pd

# Import TF-IDF Vectorizer to convert movie tags into numerical vectors.
from sklearn.feature_extraction.text import TfidfVectorizer

# Import cosine_similarity to measure similarity between movies.
from sklearn.metrics.pairwise import cosine_similarity

In [6]:
# Load the cleaned movie dataset created during EDA.
new_movie_df = pd.read_csv('cleaned_movies.csv')

In [14]:
new_movie_df

,movie_id,title,tags
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di..."
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha..."
2,206647,Spectre,A cryptic message from Bond’s past sends him o...
3,49026,The Dark Knight Rises,Following the death of District Attorney Harve...
4,49529,John Carter,"John Carter is a war-weary, former military ca..."
...,...,...,...
4804,9367,El Mariachi,El Mariachi just wants to play his guitar and ...
4805,72766,Newlyweds,A newlywed couple's honeymoon is upended by th...
4806,231617,"Signed, Sealed, Delivered","""Signed, Sealed, Delivered"" introduces a dedic..."
4807,126186,Shanghai Calling,When ambitious New York attorney Sam is sent t...


In [9]:
new_movie_df.shape

(4809, 3)

In [10]:
new_movie_df.dtypes

movie_id     int64
title       object
tags        object
dtype: object

## 2. TF-IDF Vectorization

In [11]:
# Create a TF-IDF vectorizer for converting movie tags into numerical features.
tfidf = TfidfVectorizer(max_features=5000,stop_words='english')

In [12]:
# TRANSFORM THE MOVIE TAGS
# Learn the vocabulary from the movie tags and transform each movie into a TF-IDF vector.
vectors = tfidf.fit_transform(new_movie_df['tags'])

In [13]:
# Display the shape of the TF-IDF matrix.
vectors.shape

(4809, 5000)

In [19]:
# Display the first 100 terms learned by the TF-IDF vectorizer.
tfidf.get_feature_names_out()[0:100]

array(['000', '007', '10', '100', '11', '12', '13', '14', '15', '16',
       '17', '18', '18th', '19', '1930s', '1940s', '1950s', '1960s',
       '1970s', '1980', '1980s', '1985', '1990s', '1999', '19th',
       '19thcentury', '20', '200', '2003', '2009', '20th', '21st', '23',
       '24', '25', '30', '300', '3d', '40', '50', '500', '60', '60s',
       '70', '70s', 'aaron', 'aaroneckhart', 'abandoned', 'abducted',
       'abigailbreslin', 'abilities', 'ability', 'able', 'aboard',
       'abuse', 'abusive', 'academic', 'academy', 'accept', 'accepted',
       'accepts', 'access', 'accident', 'accidental', 'accidentally',
       'accompanied', 'accomplish', 'account', 'accountant', 'accused',
       'ace', 'achieve', 'act', 'acting', 'action', 'actionhero',
       'actions', 'activist', 'activities', 'activity', 'actor', 'actors',
       'actress', 'acts', 'actual', 'actually', 'adam', 'adams',
       'adamsandler', 'adamshankman', 'adaptation', 'adapted', 'addict',
       'addicted', 'ad

In [20]:
# Display the total number of features learned by the vectorizer.
len(tfidf.get_feature_names_out())

5000

## 3. COSINE SIMILARITY

In [21]:
# Calculate the cosine similarity between every pair of movies.
similarity = cosine_similarity(vectors)

In [23]:
similarity

array([[1.        , 0.02190774, 0.01218387, ..., 0.00540869, 0.00609943,
        0.        ],
       [0.02190774, 1.        , 0.01246082, ..., 0.01728524, 0.        ,
        0.        ],
       [0.01218387, 0.01246082, 1.        , ..., 0.01623233, 0.        ,
        0.        ],
       ...,
       [0.00540869, 0.01728524, 0.01623233, ..., 1.        , 0.02911913,
        0.03276304],
       [0.00609943, 0.        , 0.        , ..., 0.02911913, 1.        ,
        0.01711901],
       [0.        , 0.        , 0.        , ..., 0.03276304, 0.01711901,
        1.        ]], shape=(4809, 4809))

In [24]:
similarity.shape

(4809, 4809)

##### Let's check with a single movie like avatar

In [25]:
# Find the index of Avatar in the movie dataframe.
movie_index = new_movie_df[new_movie_df['title'] == 'Avatar'].index[0]
movie_index

np.int64(0)

In [26]:
# Get the cosine similarity scores of Avatar with every movie.
avatar_similarity = similarity[movie_index]

In [27]:
# Check how many similarity scores we have.
len(avatar_similarity)

4809

In [29]:
# Sort movie indices according to their similarity with Avatar.
sorted_indices = sorted(enumerate(avatar_similarity), key=lambda x: x[1],reverse=True)

In [30]:
# Display the top 10 most similar movie indices and their similarity scores.
sorted_indices[:10]

[(0, np.float64(1.0)),
 (3729, np.float64(0.20357444133201313)),
 (582, np.float64(0.1973443580366979)),
 (3607, np.float64(0.18373365766286115)),
 (47, np.float64(0.17101253287806512)),
 (539, np.float64(0.1650459800039916)),
 (942, np.float64(0.16263847635970957)),
 (2405, np.float64(0.15881736196987076)),
 (1916, np.float64(0.15779102331480388)),
 (3537, np.float64(0.15314735085492293))]

In [31]:
# Display the titles corresponding to the most similar movie indices.
for index, score in sorted_indices[1:11]:
    print(new_movie_df.iloc[index]['title'], "→", round(score, 4))

Falcon Rising → 0.2036
Battle: Los Angeles → 0.1973
Apollo 18 → 0.1837
Star Trek Into Darkness → 0.171
Titan A.E. → 0.165
The Book of Life → 0.1626
Aliens → 0.1588
Lifeforce → 0.1578
Galaxina → 0.1531
Jarhead → 0.1514


## 4. Recommendation Function

In [54]:
# Create a function to return movies similar to a given movie.
def recommend(movie_title, top_n):

    # Check whether the requested movie exists in the dataset.
    if movie_title not in new_movie_df['title'].values:

        # Return a message when the movie is not found.
        return f"Movie '{movie_title}' was not found in the dataset."
    
    # Find the index of the movie entered by the user.
    movie_index = new_movie_df[new_movie_df['title'] == movie_title].index[0]

    # Get the similarity scores of the selected movie with every other movie.
    movie_similarity = similarity[movie_index]

    # Sort the movies according to their similarity scores in descending order.
    sorted_indices = sorted(enumerate(movie_similarity), key=lambda x: x[1], reverse=True)

    # Create an empty list to store the recommendations.
    recommendations = []

    # Select the top 10 similar movies, excluding the input movie itself.
    for index, score in sorted_indices[1:top_n+1]:

        # Get the title of the recommended movie.
        title = new_movie_df.iloc[index]['title']

        # Store the movie title and similarity score.
        recommendations.append({'title': title, 'similarity_score': round(score, 4)})

    # Convert the recommendations list into a DataFrame.
    return pd.DataFrame(recommendations)

In [55]:
# Test the updated recommendation function.
recommend('Avatar', top_n=5)

,title,similarity_score
0,Falcon Rising,0.2036
1,Battle: Los Angeles,0.1973
2,Apollo 18,0.1837
3,Star Trek Into Darkness,0.1710
4,Titan A.E.,0.1650


In [56]:
#Test the updated recommendation function.
recommend('Falcon Rising', top_n=5)

,title,similarity_score
0,Brother,0.2750
1,Showdown in Little Tokyo,0.2394
2,Battle: Los Angeles,0.2117
3,Avatar,0.2036
4,Jarhead,0.2012


In [57]:
recommend('Avtar', top_n=5)

"Movie 'Avtar' was not found in the dataset."

## 5. Testing the Recommendation System
**The recommendation engine is tested using representative movie titles
to verify that it produces relevant recommendations for valid inputs
and handles invalid inputs appropriately.**

##### Test 1 — Successful scenario: Avatar (SciFi)

In [67]:
# Test the recommendation system with a valid movie title.
avatar_recommendations = recommend('Avatar', top_n=5)

# Display the recommendations.
avatar_recommendations

,title,similarity_score
0,Falcon Rising,0.2036
1,Battle: Los Angeles,0.1973
2,Apollo 18,0.1837
3,Star Trek Into Darkness,0.1710
4,Titan A.E.,0.1650


##### Test 2 — Successful scenario: The Dark Knight Rises

In [68]:
# Test the recommendation system with another valid movie title.
dark_knight_recommendations = recommend('The Dark Knight Rises',top_n=5)

# Display the recommendations.
dark_knight_recommendations

,title,similarity_score
0,The Dark Knight,0.4560
1,Batman Returns,0.4002
2,Batman Begins,0.3536
3,Batman Forever,0.3367
4,Batman,0.3312


##### Test 3 — Successful scenario: Titanic (romantic/drama)

In [69]:
# Test the recommendation system with Titanic.
titanic_recommendations = recommend('Titanic', top_n=5)

# Display the recommendations.
titanic_recommendations

,title,similarity_score
0,Ghost Ship,0.2173
1,Poseidon,0.2070
2,In the Heart of the Sea,0.1954
3,Triangle,0.1889
4,Pirates of the Caribbean: On Stranger Tides,0.1858


##### Test 4 — Successful scenario: Toy Story (animated/family drama)

In [70]:
# Test the recommendation system with Toy Story.
toy_story_recommendations = recommend('Toy Story', top_n=5)

# Display the recommendations.
toy_story_recommendations

,title,similarity_score
0,Toy Story 3,0.5137
1,Toy Story 2,0.4947
2,The 40 Year Old Virgin,0.3446
3,Class of 1984,0.1784
4,Factory Girl,0.1758


##### Test 5 — Failure scenario: Invalid title

In [66]:
# Test how the system handles a movie title that does not exist.
invalid_recommendation = recommend('Avtar', top_n=5)

# Display the result.
invalid_recommendation

"Movie 'Avtar' was not found in the dataset."

##### Test 6 — Failure scenario: Empty input

In [72]:
# Test how the system handles an empty movie title.
empty_recommendation = recommend('',top_n = 5)

# Display the result.
empty_recommendation

"Movie '' was not found in the dataset."

In [74]:
# Test the recommendation system with Inception.
inception_recommendations = recommend('Inception', top_n=5)

# Display the recommendations.
inception_recommendations

,title,similarity_score
0,Don Jon,0.1728
1,Premium Rush,0.1504
2,Cypher,0.1452
3,Hesher,0.1353
4,Duplex,0.1308


## 6. Evaluation

The baseline content-based recommendation system is evaluated using
representative movie queries, catalogue coverage, recommendation
diversity, response latency, and failure-case analysis.

Since the dataset does not contain explicit user preferences,
ratings, clicks, watch history, or relevance labels, standard
ranking metrics such as Precision@K and Recall@K cannot be directly
calculated without introducing additional assumptions.

Therefore, the evaluation focuses on metrics and tests that can be
reliably calculated from the available movie metadata and the
recommendation output.

### 6.1 Recommendation Relevance

Representative movies from different content categories were tested
to determine whether the generated recommendations are semantically
and thematically reasonable.

In [75]:
# Store the representative test cases and our observations.
evaluation_cases = pd.DataFrame({
    'movie': [
        'Avatar',
        'The Dark Knight Rises',
        'Toy Story',
        'Titanic',
        'Inception'
    ],
    'observation': [
        'Reasonable science-fiction/action recommendations.',
        'Strong Batman and related superhero recommendations.',
        'Strong recommendations, especially Toy Story 2 and Toy Story 3.',
        'Mixed recommendations; maritime/disaster similarity dominates.',
        'Weak recommendations; several results do not strongly match the core themes.'
    ]
})

# Display the evaluation observations.
evaluation_cases

,movie,observation
0,Avatar,Reasonable science-fiction/action recommendati...
1,The Dark Knight Rises,Strong Batman and related superhero recommenda...
2,Toy Story,"Strong recommendations, especially Toy Story 2..."
3,Titanic,Mixed recommendations; maritime/disaster simil...
4,Inception,Weak recommendations; several results do not s...


### 6.2 Catalogue Coverage

Catalogue coverage measures how much of the available movie catalogue
appears in the recommendation results.

Higher coverage indicates that the recommendation system is capable
of surfacing a broader portion of the catalogue rather than repeatedly
recommending the same small group of movies.`

In [76]:
# Define representative movies for the coverage analysis.
test_movies = [
    'Avatar',
    'The Dark Knight Rises',
    'Titanic',
    'Toy Story',
    'Inception'
]

# Store recommended movie titles from all test cases.
recommended_movies = set()

# Generate recommendations for each test movie.
for movie in test_movies:

    # Get the top 10 recommendations.
    recommendations = recommend(movie, top_n=10)

    # Add the recommended movie titles to the set.
    if isinstance(recommendations, pd.DataFrame):
        recommended_movies.update(recommendations['title'])

In [77]:
# Calculate the number of unique movies recommended.
unique_recommendations = len(recommended_movies)

# Calculate the total number of movies in the catalogue.
total_movies = len(new_movie_df)

# Calculate catalogue coverage as a percentage.
coverage = (unique_recommendations / total_movies) * 100

# Display the coverage results.
print("Unique movies recommended:", unique_recommendations)
print("Total movies in catalogue:", total_movies)
print("Catalogue coverage:", round(coverage, 2), "%")

Unique movies recommended: 49
Total movies in catalogue: 4809
Catalogue coverage: 1.02 %


### 6.3 Recommendation Diversity

Diversity measures how different the recommended movies are from one
another. A recommendation system that repeatedly returns very similar
items may provide limited choice to the user.

In [78]:
# Calculate the total number of recommendation slots.
total_recommendations = len(test_movies) * 10

# Calculate the percentage of unique recommended titles.
diversity = (unique_recommendations / total_recommendations) * 100

# Display the diversity result.
print("Total recommendation slots:", total_recommendations)
print("Unique recommended movies:", unique_recommendations)
print("Recommendation diversity:", round(diversity, 2), "%")

Total recommendation slots: 50
Unique recommended movies: 49
Recommendation diversity: 98.0 %


### 6.4 Recommendation Latency

Recommendation latency measures the time required to generate
recommendations after a movie has been selected.

In [80]:
import time

In [81]:
# Measure the time required to generate recommendations.
start_time = time.perf_counter()

# Generate recommendations for a representative movie.
recommend('Avatar', top_n=10)

# Calculate elapsed time.
latency = time.perf_counter() - start_time

# Display the latency.
print("Recommendation latency:", round(latency, 6), "seconds")

Recommendation latency: 0.028261 seconds


### 6.5 Failure Scenarios

The system is also tested using invalid and empty movie titles to
verify that the recommendation function handles invalid input without
producing an unhandled exception.

In [83]:
# Test an invalid movie title.
invalid_result = recommend('Avtar', top_n =5)

# Display the result.
print("Invalid input result:")
print(invalid_result)

Invalid input result:
Movie 'Avtar' was not found in the dataset.


In [84]:
# Test an empty movie title.
empty_result = recommend('',top_n =5)

# Display the result.
print("Empty input result:")
print(empty_result)

Empty input result:
Movie '' was not found in the dataset.


### 6.6 Baseline Evaluation Summary

The baseline content-based recommender successfully generates
recommendations for valid movie titles and handles invalid or empty
inputs without crashing.

The qualitative tests show strong recommendations for some movies,
such as The Dark Knight Rises and Toy Story, while weaker results are
observed for movies such as Titanic and Inception.

The system therefore demonstrates a functional baseline but also
shows the limitations of relying solely on lexical similarity from
TF-IDF representations.

## 7. Findings

The baseline content-based recommendation system successfully generates
movie recommendations using TF-IDF representations of movie metadata
and cosine similarity.

The following findings were observed during testing:

- The system produced strong recommendations for movies with distinctive
  and closely related content, such as *The Dark Knight Rises* and
  *Toy Story*.
- For *The Dark Knight Rises*, several Batman-related movies received
  high similarity scores, indicating that the model successfully captured
  shared content characteristics.
- For *Toy Story*, *Toy Story 2* and *Toy Story 3* were ranked among the
  highest recommendations, demonstrating strong content overlap.
- Some movies produced weaker recommendations. For example, the results
  for *Titanic* were influenced strongly by maritime and disaster-related
  terms, while the results for *Inception* did not consistently capture
  its complete psychological and science-fiction themes.
- Invalid and empty movie-title inputs were handled without an unhandled
  exception.
- The recommendation system provides a functional baseline for
  content-based movie recommendation using only movie metadata.

## 8. Limitations

The current recommendation system has several limitations:

1. **Lexical rather than semantic similarity**

   TF-IDF primarily measures the importance and overlap of individual
   terms. It does not fully understand the semantic meaning of a movie's
   story or themes.

2. **Dependence on available metadata**

   Recommendation quality depends on the quality and completeness of
   the movie genres, keywords, overview, cast, and crew information.

3. **No user preference information**

   The current system is content-based and does not use user ratings,
   watch history, clicks, likes, or other personalized behaviour.

4. **Limited personalization**

   Two users entering the same movie receive the same recommendations
   because the system does not currently model individual user
   preferences.

5. **Duplicate or similar movie titles**

   Movies with identical or very similar titles can make title-based
   selection ambiguous.

6. **Popularity and cold-start limitations**

   The system does not explicitly model popularity or user behaviour,
   and recommendations are based primarily on the available movie
   metadata.

7. **Recommendation quality varies across movies**

   Testing showed that some movies receive highly relevant
   recommendations while others receive weaker results. This reflects
   the limitations of using TF-IDF and cosine similarity as the sole
   recommendation strategy.

## 9. Future Improvements

The baseline system can be further improved through the following
approaches:

1. **Bag-of-Words comparison**

   Compare the current TF-IDF representation with a Bag-of-Words
   representation to determine which representation produces better
   recommendation results.

2. **Improved NLP preprocessing**

   Apply techniques such as stemming or lemmatization to reduce the
   effect of different word forms such as "actor" and "actors".

3. **Semantic embeddings**

   Experiment with word embeddings or sentence/document embeddings to
   capture semantic relationships that TF-IDF may miss.

4. **Field-aware weighting**

   Assign different importance to genres, keywords, overview, cast,
   and crew instead of treating all information in the combined tags
   equally.

5. **Hybrid recommendation**

   Combine content-based recommendations with collaborative filtering
   when user ratings, interactions, or watch-history data become
   available.

6. **Personalization**

   Incorporate user preferences, ratings, watch history, likes, and
   dislikes to provide personalized recommendations.

7. **Diversity-aware re-ranking**

   Introduce diversity constraints so that the recommendation list does
   not contain overly similar movies.

8. **Improved title selection**

   Use a searchable or dropdown-based movie selector in the application
   to reduce ambiguity caused by duplicate movie titles.

9. **More comprehensive evaluation**

   With user interaction or relevance data, evaluate the system using
   metrics such as Precision@K, Recall@K, MAP, NDCG, coverage, diversity,
   and other appropriate recommendation metrics.

10. **Production optimization**

    Separate model training from recommendation inference and persist
    reusable model artifacts so that the application does not need to
    recompute the complete similarity matrix for every user session.

## 10. Conclusion

The CineMovie baseline recommendation system successfully implements a
content-based recommendation pipeline using movie metadata.

The workflow consists of:

Movie metadata
→ Data preprocessing
→ Combined movie tags
→ TF-IDF vectorization
→ Cosine similarity
→ Top-N recommendations

The baseline demonstrates that textual movie metadata can be used to
generate meaningful recommendations without requiring user-rating or
interaction data.

Testing also identified important limitations, particularly for movies
whose semantic meaning cannot be fully captured through lexical
similarity. These limitations provide clear directions for future
improvements such as Bag-of-Words comparison, semantic embeddings,
field-aware weighting, personalization, and hybrid recommendation
methods.